In [41]:
import pandas as pd
import numpy as np

df = pd.read_csv('../data/raw/calories.csv')

print(df.shape)
df.head()

(15000, 9)


,User_ID,Gender,Age,Height,Weight,Duration,Heart_Rate,Body_Temp,Calories
0,14733363,male,68,190.0,94.0,29.0,105.0,40.8,231.0
1,14861698,female,20,166.0,60.0,14.0,94.0,40.3,66.0
2,11179863,male,69,179.0,79.0,5.0,88.0,38.7,26.0
3,16180408,female,34,179.0,71.0,13.0,100.0,40.5,71.0
4,17771927,female,27,154.0,58.0,10.0,81.0,39.8,35.0


In [42]:
df_processed = df.copy()

df_processed = df_processed.drop(columns=['User_ID'])

df_processed.head()

,Gender,Age,Height,Weight,Duration,Heart_Rate,Body_Temp,Calories
0,male,68,190.0,94.0,29.0,105.0,40.8,231.0
1,female,20,166.0,60.0,14.0,94.0,40.3,66.0
2,male,69,179.0,79.0,5.0,88.0,38.7,26.0
3,female,34,179.0,71.0,13.0,100.0,40.5,71.0
4,female,27,154.0,58.0,10.0,81.0,39.8,35.0


In [43]:
df = pd.read_csv('../data/processed/processed_fitness.csv')

In [44]:
df.columns

Index(['User_ID', 'Gender', 'Age', 'Height', 'Weight', 'Duration',
       'Heart_Rate', 'Body_Temp', 'Calories', 'Workout_Intensity'],
      dtype='object')

In [45]:
df_processed = df.copy()

df_processed = df_processed.drop(columns=['User_ID'], errors='ignore')

In [46]:
df_processed['Gender'] = df_processed['Gender'].map({
    'female': 0,
    'male': 1
})

df_processed.head()

,Gender,Age,Height,Weight,Duration,Heart_Rate,Body_Temp,Calories,Workout_Intensity
0,1,68,190.0,94.0,29.0,105.0,40.8,231.0,High
1,0,20,166.0,60.0,14.0,94.0,40.3,66.0,Medium
2,1,69,179.0,79.0,5.0,88.0,38.7,26.0,Low
3,0,34,179.0,71.0,13.0,100.0,40.5,71.0,Medium
4,0,27,154.0,58.0,10.0,81.0,39.8,35.0,Low


In [47]:
df_processed['Gender'].value_counts()

Gender
0    7553
1    7447
Name: count, dtype: int64

In [48]:
intensity_mapping = {
    'Low': 0,
    'Medium': 1,
    'High': 2
}

df_processed['Workout_Intensity'] = df_processed['Workout_Intensity'].map(intensity_mapping)

df_processed.head()

,Gender,Age,Height,Weight,Duration,Heart_Rate,Body_Temp,Calories,Workout_Intensity
0,1,68,190.0,94.0,29.0,105.0,40.8,231.0,2
1,0,20,166.0,60.0,14.0,94.0,40.3,66.0,1
2,1,69,179.0,79.0,5.0,88.0,38.7,26.0,0
3,0,34,179.0,71.0,13.0,100.0,40.5,71.0,1
4,0,27,154.0,58.0,10.0,81.0,39.8,35.0,0


In [49]:
df_processed['Workout_Intensity'].value_counts()

Workout_Intensity
1    7503
0    3798
2    3699
Name: count, dtype: int64

In [50]:
# we are removing calories from iput feature X because- workout intensity is directly engineered from calories, so introducing them would induce data leaking
X = df_processed[
    [
        'Gender',
        'Age',
        'Height',
        'Weight',
        'Duration',
        'Heart_Rate',
        'Body_Temp'
    ]
]

y = df_processed['Workout_Intensity']

In [51]:
print(X.shape)
print(y.shape)

(15000, 7)
(15000,)


In [52]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [53]:
print(X_train.shape)
print(X_test.shape)

print(y_train.value_counts(normalize=True))
print(y_test.value_counts(normalize=True))

(12000, 7)
(3000, 7)
Workout_Intensity
1    0.500250
0    0.253167
2    0.246583
Name: proportion, dtype: float64
Workout_Intensity
1    0.500000
0    0.253333
2    0.246667
Name: proportion, dtype: float64


In [54]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)

X_test_scaled = scaler.transform(X_test)

In [55]:
import pandas as pd

pd.DataFrame(X_train_scaled).describe()

,0,1,2,3,4,5,6
count,1.200000e+04,1.200000e+04,1.200000e+04,1.200000e+04,1.200000e+04,1.200000e+04,1.200000e+04
mean,-1.000681e-16,-1.255292e-16,-7.638334e-17,-3.552714e-18,4.115227e-17,-1.432928e-16,7.490009e-15
std,1.000042e+00,1.000042e+00,1.000042e+00,1.000042e+00,1.000042e+00,1.000042e+00,1.000042e+00
min,-9.917011e-01,-1.337424e+00,-3.600524e+00,-2.449244e+00,-1.745499e+00,-2.972517e+00,-3.751278e+00
25%,-9.917011e-01,-8.668125e-01,-7.298417e-01,-7.895872e-01,-9.039613e-01,-7.809277e-01,-5.449114e-01
50%,-9.917011e-01,-2.197217e-01,4.034125e-02,-5.933826e-02,5.779558e-02,-5.039786e-02,2.246167e-01
75%,1.008368e+00,7.803278e-01,7.405075e-01,8.036832e-01,8.993328e-01,7.844934e-01,7.376354e-01
max,1.008368e+00,2.133336e+00,3.331123e+00,3.525520e+00,1.740870e+00,3.393529e+00,1.891928e+00


In [56]:
# in the above .describe function, we should always check for Mean ~ 0 and standard_deviation ~ 1.Then we can assume our scaling worked correctly

In [57]:
# We have used .fit in only training data and not on test data because what .fit does is it learns statistics of our data like mean and stand deviation.
# we have not done this in case of test data because we don't want test data to learn statitics, that can cause data leaking
#question- Why fit on training data but only transform test data?
# ans- We fit the scaler only on the training data so that the mean and standard deviation are learned exclusively from the training set. 
# We then use those same statistics to transform the test set. This prevents data leakage and ensures the test data remains completely 
# unseen during training.

In [58]:
import tensorflow as tf

print(tf.__version__)

2.21.0


In [59]:
# Build the arcitecture of the model
model = tf.keras.Sequential([
    tf.keras.layers.Dense(
        32,
        activation='relu',
        input_shape=(7,)
    ),

    tf.keras.layers.Dense(
        16,
        activation='relu'
    ),

    tf.keras.layers.Dense(
        3,
        activation='softmax'
    )
])

/Users/adarshpatkar/fitsense-ai/.venv/lib/python3.10/site-packages/keras/src/layers/core/dense.py:95: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [60]:
model.summary()

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense_3 (Dense)                 │ (None, 32)             │           256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 16)             │           528 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ (None, 3)              │            51 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 835 (3.26 KB)

 Trainable params: 835 (3.26 KB)

 Non-trainable params: 0 (0.00 B)

In [61]:
#what each terms means-
# here optmizer decides how should weight and biases should be added.
# adam- Adam is one of the most popular optimizers because it automatically adjusts learning rates and usually works well without much 
# tuning (think of it as gradient descend)
# Loss function measures how wrong was our predictions
# Why sparse_categorical_crossentropy- because our labels are 0,1,2 and not [1, 0, 0] [0, 1, 0] etc << called integer encoding
# choose categorical cross entropy when label looks like - [1, 0, 0]- low, [0, 1, 0] - medium
# metricts - accurcy- beside loss, also show classification accuracy
# model.complile-- tells model all this above things

model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

In [62]:
print("Model compiled successfully!")

Model compiled successfully!


In [63]:
# next we will train the model, so what does each terms means-
#epochs- 20 << epochs-1 means see all 12, 0000 training example once, then epochs-2 means see all 12,000 training ex again.
#validation_split = 0.2<< tensorflow takes 80% actual training, 20 % for validation from X_train
# batch_size = 32 << updates weights in batches of 32 << takes first 32 rows- update weights, then 32 rows- update weights < also called mini-batch gradient descent

In [64]:
history = model.fit(
    X_train_scaled,
    y_train,
    epochs=20,
    validation_split=0.2,
    batch_size=32,
    verbose=1
)

Epoch 1/20
300/300 ━━━━━━━━━━━━━━━━━━━━ 1s 609us/step - accuracy: 0.8716 - loss: 0.4596 - val_accuracy: 0.9287 - val_loss: 0.2128
Epoch 2/20
300/300 ━━━━━━━━━━━━━━━━━━━━ 0s 369us/step - accuracy: 0.9374 - loss: 0.1688 - val_accuracy: 0.9488 - val_loss: 0.1395
Epoch 3/20
300/300 ━━━━━━━━━━━━━━━━━━━━ 0s 368us/step - accuracy: 0.9568 - loss: 0.1190 - val_accuracy: 0.9671 - val_loss: 0.1003
Epoch 4/20
300/300 ━━━━━━━━━━━━━━━━━━━━ 0s 361us/step - accuracy: 0.9672 - loss: 0.0959 - val_accuracy: 0.9721 - val_loss: 0.0827
Epoch 5/20
300/300 ━━━━━━━━━━━━━━━━━━━━ 0s 385us/step - accuracy: 0.9722 - loss: 0.0805 - val_accuracy: 0.9762 - val_loss: 0.0723
Epoch 6/20
300/300 ━━━━━━━━━━━━━━━━━━━━ 0s 361us/step - accuracy: 0.9768 - loss: 0.0699 - val_accuracy: 0.9762 - val_loss: 0.0680
Epoch 7/20
300/300 ━━━━━━━━━━━━━━━━━━━━ 0s 365us/step - accuracy: 0.9781 - loss: 0.0621 - val_accuracy: 0.9846 - val_loss: 0.0566
Epoch 8/20
300/300 ━━━━━━━━━━━━━━━━━━━━ 0s 362us/step - accuracy: 0.9797 - loss: 0.0560 - 

In [65]:
test_loss, test_accuracy = model.evaluate(
    X_test_scaled,
    y_test,
    verbose=1
)

print("Test Loss:", test_loss)
print("Test Accuracy:", test_accuracy)

94/94 ━━━━━━━━━━━━━━━━━━━━ 0s 321us/step - accuracy: 0.9920 - loss: 0.0272 
Test Loss: 0.027219180017709732
Test Accuracy: 0.9919999837875366


In [66]:
# this is confusion matrix code which tells us how many classed were confused with each other

import numpy as np

y_pred = model.predict(X_test_scaled)

y_pred_classes = np.argmax(y_pred, axis=1)

from sklearn.metrics import confusion_matrix

cm = confusion_matrix(y_test, y_pred_classes)

cm

94/94 ━━━━━━━━━━━━━━━━━━━━ 0s 339us/step


array([[ 758,    2,    0],
       [  12, 1484,    4],
       [   0,    6,  734]])

In [67]:
# How to read it:
# Rows- Truth value, row1: actual low workout, row2: actual medium workout, row3: actual high workout

# diagonal- correct predictions, 746: correctly identified as low, 1487: correctly identified as medium, 737: correctly identified as high

# where did the model made mistakes-
# everything outside diagonals
# 746: correctly predicted as low, 14: wrongly predicted as medium, 0: wrongly predcited as High, so matrix didn't confused low with High.
# similarly, 4: medium workout classified as low, 1487 correct and 9: medium workout classified as high

In [68]:
# "What did you learn from your confusion matrix?"
#The model achieved 99% test accuracy. The confusion matrix showed that most predictions lie on the diagonal, indicating correct 
# classification. The model rarely confused low and high intensity workouts, which suggests the classes are well separated. 
# Most misclassifications occurred between medium intensity and its neighboring classes, which is expected because medium intensity 
# represents a boundary region between low and high intensity workouts.

In [69]:
# remembering precision
# when I predict high, how often am I right

# remembering recall
# of all the high workout that exists, how many did I catch

In [70]:
from sklearn.metrics import classification_report

print(
    classification_report(
        y_test,
        y_pred_classes,
        target_names=['Low', 'Medium', 'High']
    )
)

              precision    recall  f1-score   support

         Low       0.98      1.00      0.99       760
      Medium       0.99      0.99      0.99      1500
        High       0.99      0.99      0.99       740

    accuracy                           0.99      3000
   macro avg       0.99      0.99      0.99      3000
weighted avg       0.99      0.99      0.99      3000



In [71]:
# understanding report
# support: Num of samples of low, medium and high workout
# precision: Low- 0.99 , when model precicts low, it is correct 99% of the time
# recall: Low- 0.98, Out of the actual low workout, model successfully found 98% of them
#F1-score: Metrics combine precision and recall into single num, when both are high f1~1 and when both are low it drops
# for our case it's 0.99 so that's good
#macro-average: treats every class equally regardless of samples counts
#weighted-average: Weights each class by how many examples it has.

In [72]:
#question-
#How did your model perform?
# The neural network achieved 99% test accuracy. Precision, recall, and F1-score were approximately 0.99 for all three workout 
# intensity classes. The confusion matrix showed very few misclassifications, mainly between Medium and neighboring classes, 
# indicating strong generalization and balanced performance across categories.

In [73]:
model

<Sequential name=sequential_1, built=True>

In [74]:
import os

print("Model size:", os.path.getsize("../models/best_model.keras"))
print("Scaler size:", os.path.getsize("../models/scaler.pkl"))

print(type(scaler))

Model size: 35440
Scaler size: 1087
<class 'sklearn.preprocessing._data.StandardScaler'>


In [75]:
model.summary()

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense_3 (Dense)                 │ (None, 32)             │           256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 16)             │           528 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ (None, 3)              │            51 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,507 (9.80 KB)

 Trainable params: 835 (3.26 KB)

 Non-trainable params: 0 (0.00 B)

 Optimizer params: 1,672 (6.54 KB)

In [76]:
model.save("../models/best_model.keras")

print("Model saved!")

Model saved!


In [77]:
import joblib

joblib.dump(
    scaler,
    "../models/scaler.pkl"
)

print("Scaler saved!")

Scaler saved!


In [78]:
import os

print("Model size:", os.path.getsize("../models/best_model.keras"))
print("Scaler size:", os.path.getsize("../models/scaler.pkl"))

Model size: 35446
Scaler size: 1087


In [79]:
# we have now saved the model as well scaler, both of the value now contains-
#best_model.keras now contains:

#Neural network architecture
#Learned weights (W)
#Learned biases (b)
#Optimizer state

#scaler.pkl now contains:

#Mean of every feature
#Standard deviation of every feature
#StandardScaler object

In [80]:
sample = X_test_scaled[:1]

print(sample.shape)

pred = model.predict(sample)

print(pred)

(1, 7)
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step
[[9.9992168e-01 7.8298814e-05 2.8206201e-11]]
